(DebrisCylinder)=
# Debris in potential flow around a cylinder

Another example to illustrate the use of the `debris_tracking` module.  Still under development.

Passive advection in a given velocity field defined by `u(x,y,t), v(x,y,t)` that are independent of time `t` in the example shown here.

Running this requires `shapely` and `clawpack`, the latter for the `animation_tools` that are further illustrated in this [demo notebook](https://www.clawpack.org/gallery/_static/apps/notebooks/visclaw/animation_tools_demo.html).

See [../../docs/python_environment.md](python_environment).


In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from importlib import reload
import shapely
from shapely import plotting as shapely_plotting
from clawpack.visclaw import animation_tools
from IPython.display import display, HTML


In [ ]:
import sys
sys.path.insert(0,'../../src')

In [ ]:
from geoclaw_debris import debris_tracking
reload(debris_tracking) # for debugging

## Some convenience functions for plotting:

In [ ]:
def plot_quiver(extent, dx, dy, u, v):
    """
    Make a quiver plot of a velocity field defined by function u(x,y), v(x,y)
    Plots over the extent [x1,x2,y1,y2] on a grid with spacing dx,dy.
    Scale the arrow lengths based on the maximum velocities and mesh spacing.
    """
    xvel = arange(extent[0], extent[1]+dx/2, dx)
    yvel = arange(extent[2], extent[3]+dy/2, dy)
    Xvel,Yvel = meshgrid(xvel,yvel,indexing='xy')
    Uvel = u(Xvel,Yvel,0)
    Vvel = v(Xvel,Yvel,0)
    smax = sqrt(Uvel**2 + Vvel**2).max()
    #print(f'Scaling by 1.1*smax/dx = {1.1*smax/dx}')
    quiver(Xvel,Yvel,Uvel,Vvel,scale=1.1*smax/dx,scale_units='xy',color='gray',width=0.003)

## Flow around cylinder


Note that u,v are define as functions of `(x,y,t)` and can be time dependent, but the example in this notebook  uses a velocity field that is constant in time.

The flow field is
$$
u(x,y) &= U + 2(R^2 - x^2)/(x^2 + y^2)^2,\\
v(x,y) &= -2Uxy R^2/(x^2 + y^2)^2.
$$
This can be computed as $(u,v) = \nabla \phi$ for the velocity potential
$$
\phi(x,y) = Ux + R^2 x/(x^2 + y^2).
$$
or it can be computed from the stream function
$$
\psi(r,\theta) = U(r - R^2/r) \sin\theta
$$
(expressed in polar coordinates).

In [ ]:
# potential flow around cylinder:
U = 1
R = 2
r = lambda x,y: sqrt(x**2 + y**2)

def u(x,y,t):
    r2 = array(x**2 + y**2)
    one_over_r4 = divide(1, r2**2, where=r2>R**2, out=zeros(r2.shape))
    u = where(r2>R**2,U, 0) + 2*(R**2 - x**2)*one_over_r4
    return u

def v(x,y,t):
    r2 = array(x**2 + y**2)
    one_over_r4 = divide(1, r2**2, where=r2>R**2, out=zeros(r2.shape))
    v = -U*(R**2 * 2*x*y) * one_over_r4
    return v
    

In [ ]:
# quiver plot of flow field:
plot_quiver([-5,5,-4,4], .5, 0.5, u, v)

# plot cylinder:
thetaR = linspace(0, 2*pi, 100)
xR = R*cos(thetaR)
yR = R*sin(thetaR)
plot(xR,yR,'b')

axis('equal')
title('Potential flow around a cylinder');

## Function to make an animation of debris motion


In [ ]:
def make_anim(debris_path_list, extent, figsize=(6,6), obst_list=[], domain=None, u=None, v=None):
    """
    :Inputs:
        - debris_path_list: a list of debris_tools.DebrisPath objects describing the path in time
            of each debris object.  This must first be computed and passed in.
        - extent: axis [x1,x2,y1,y2] for plots
        - figsize: figure size
        - obst_list: list of obstacles that were specified when computing the debris paths, a list
            of dictionaries (for adding the obstacles to the plots)
        - domain: the domain specified when computing the debris paths, a rectangle [x1,x2,y1,y2] bounded
            by solid walls the debris objects could not penetrate (for plotting these walls).
            domain == None is equivalent to domain == [-inf, inf, -inf, inf] (no walls on any side)
        - u,v: functions of (x,y,t) specifying the velocity field. If passed in, quiver plot will be added.
    """
    
    figs = [] # to accumulate frames of animation
    for n in range(nsteps+1):
        fig = figure(figsize=figsize)

        if (u is not None) and (v is not None):
            # quiver plot of velocity field:
            x1,x2,y1,y2 = extent
            if domain is not None:
                x1 = max(x1,domain[0])
                x2 = min(x2,domain[1])
                y1 = max(y1,domain[2])
                y2 = min(y2,domain[3])
            dx = (x2-x1)/20
            dy = (y2-y1)/20
            plot_quiver([x1,x2,y1,y2], dx, dy, u, v)

        # plot any obstacles:
        for obst in obst_list:
            shapely.plotting.plot_polygon(obst['polygon'], add_points=False,
                                          color='blue', alpha=0.5)

        if domain is not None:
            # plot any walls:
            x1,x2,y1,y2 = extent
            x1wall,x2wall,y1wall,y2wall = domain
            plot([x1wall,x1wall], [max(y1wall,y1),min(y2wall,y2)], 'g')
            plot([x2wall,x2wall], [max(y1wall,y1),min(y2wall,y2)], 'g')
            plot([max(x1wall,x1),min(x2wall,x2)], [y1wall,y1wall], 'g')
            plot([max(x1wall,x1),min(x2wall,x2)], [y2wall,y2wall], 'g')


        # plot debris paths:
        for debris_path in debris_path_list:
            debris = debris_path.debris
            t_n = debris_path.times[n]
            z_n = debris_path.z_path[n]
            xc,yc = debris.get_corners(z_n, close_poly=True)
            plot(xc, yc, 'b')
            
        #grid(True)
        axis(extent)
        title(f'Time {t_n} seconds')
        gca().set_aspect(1)
        figs.append(fig)
        close(fig)

    images = animation_tools.make_images(figs)
    anim = animation_tools.animate_images(images, figsize=figsize)
    return anim

## Debris objects in flow

The next cell takes 10 minutes or so to run as set up with 8 objects, one obstacle, and two walls.

In [ ]:
debris_list = []

for xd in [-4, -6]:
    for yd in arange(-0.1, 2, 0.65):
        #[-0.2, 0.45, 1.1, 1.75]:
        debris = debris_tracking.DebrisObject()
        debris.L = [1.5,0.5,1.5]
        debris.phi = [pi/2, pi/2, pi/2]
        debris.z = (xd, yd, 0)
        debris.advect = True  # Particle advected with flow
        debris_list.append(debris)


z0_list = [db.z for db in debris_list]

h = lambda x,y,t: 10.

# ---------------------------------
# obstacles:

def make_circular_obstacle(x0,y0,R):
    """                   
    Create obst representing a circular stationary obstacle with specified
    center and radius.
    """
    thetaR = linspace(0, 2*pi, 20)
    obst_x = x0 + R*cos(thetaR)
    obst_y = y0 + R*sin(thetaR)
    obst_p = vstack((obst_x,obst_y)).T
    obst_polygon = shapely.Polygon(obst_p)
    obst = {}
    obst['polygon'] = obst_polygon
    obst['xcentroid'] = x0
    obst['ycentroid'] = y0
    obst['radius'] = R
    return obst


obst = make_circular_obstacle(0,0,R)
obst_list = [obst]

# ---------------------------------
# walls:  Try it with and without

if 1:
    ywall = 4
    domain = [-inf, inf, -ywall, ywall]
else:
    domain = None  # no physical domain walls

# ---------------------------------
# time stepping parameters:

t0 = 0.
nsteps = 30
dt = 0.5

# main computational loop -- compute all the debris paths:

debris_path_list = debris_tracking.make_debris_path_list(debris_list, z0_list,
                                                         obst_list, domain,
                                                         t0,dt,nsteps,h,u,v)


# make animation:

anim = make_anim(debris_path_list, [-6,6,-4,4], (6,4), obst_list, domain, u, v)
HTML(anim.to_jshtml())

### Make plot of one object at all times:

In [ ]:
debris_path = debris_path_list[0]  # plot only one debris object

figure(figsize=(8,8))
plot_quiver([-5,5,-4,4], .5, 0.5, u, v)
c = 10*['k','r','b','g']
for n in range(nsteps+1):
    t_n = debris_path.times[n]
    z_n = debris_path.z_path[n]
    xc,yc = debris.get_corners(z_n, close_poly=True)
    plot(xc, yc, color=c[n])
plot(xR,yR,'k')
gca().set_aspect(1)  # aspect ratio for plots
axis([-6,6,-4,4])
title('Location of first object at all time steps');